# Model Demo — All VIX×EPU Combinations
### Test v1 and v2 GPT-2 models locally

Run sections independently — each loads its own model.
Make sure `booth_results/` and `booth_results_v2/` are in the same directory as this notebook.

## Shared Imports

In [1]:
import os, json, torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

BASE = os.path.dirname(os.path.abspath('')) if '__file__' not in dir() else os.path.dirname(os.path.abspath(__file__))

def load_model(model_dir):
    tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
    with open(os.path.join(model_dir, 'tokenizer_config.json')) as f:
        tcfg = json.load(f)
    special = tcfg.get('extra_special_tokens', [])
    tokenizer.add_special_tokens({'additional_special_tokens': special})
    tokenizer.pad_token = tokenizer.eos_token
    model = GPT2LMHeadModel.from_pretrained(model_dir)
    model.eval()
    device = torch.device(
        'cuda' if torch.cuda.is_available() else
        'mps'  if torch.backends.mps.is_available() else 'cpu'
    )
    model.to(device)
    print(f'Loaded {model_dir} on {device}')
    return tokenizer, model, device

def generate(tokenizer, model, device, vix_label, epu_label, topic_id,
             max_new_tokens=80, temperature=0.85, top_p=0.92):
    prompt    = f'[{vix_label}] [{epu_label}] [TOPIC_{topic_id}]'
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    prompt_len = input_ids.shape[1]
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=top_p,
            no_repeat_ngram_size=3,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0][prompt_len:], skip_special_tokens=True).strip()

print('Shared utilities ready.')

/Users/school/miniconda3/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Shared utilities ready.


---
## V1 Model — 4 Topics · 3-Bucket VIX/EPU

Loads from `booth_results/models/trump_gpt2/`

In [2]:
MODEL_DIR_V1   = 'booth_results/models/trump_gpt2'
CUTOFFS_V1     = 'booth_results/results/bucket_cutoffs.json'

tok1, mdl1, dev1 = load_model(MODEL_DIR_V1)

with open(CUTOFFS_V1) as f:
    cuts1 = json.load(f)

VIX_LABELS_V1 = ['VIX_LOW', 'VIX_MED', 'VIX_HIGH']
EPU_LABELS_V1 = ['EPU_LOW', 'EPU_MED', 'EPU_HIGH']
TOPICS_V1     = [0, 1, 2, 3]

print(f'VIX cutoffs: {[round(x,2) for x in cuts1["vix"]]}')
print(f'EPU cutoffs: {[round(x,2) for x in cuts1["epu"]]}')

/Users/school/miniconda3/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loaded booth_results/models/trump_gpt2 on mps
VIX cutoffs: [13.83, 19.32]
EPU cutoffs: [119.78, 163.24]


In [3]:
out_path = 'demo_output_v1.txt'
lines = []
lines.append('=' * 70)
lines.append('V1 MODEL — ALL VIX x EPU COMBINATIONS')
lines.append('=' * 70)

for vl in VIX_LABELS_V1:
    for el in EPU_LABELS_V1:
        lines.append(f'\n{"─"*60}')
        lines.append(f'  [{vl}] [{el}]')
        lines.append(f'{"─"*60}')
        for t in TOPICS_V1:
            text = generate(tok1, mdl1, dev1, vl, el, t)
            lines.append(f'  [Topic {t}] {text}')
        print(f'  Done: [{vl}] [{el}]')

with open(out_path, 'w') as f:
    f.write('\n'.join(lines))
print(f'\nSaved to {out_path}')

  Done: [VIX_LOW] [EPU_LOW]
  Done: [VIX_LOW] [EPU_MED]
  Done: [VIX_LOW] [EPU_HIGH]
  Done: [VIX_MED] [EPU_LOW]
  Done: [VIX_MED] [EPU_MED]
  Done: [VIX_MED] [EPU_HIGH]
  Done: [VIX_HIGH] [EPU_LOW]
  Done: [VIX_HIGH] [EPU_MED]
  Done: [VIX_HIGH] [EPU_HIGH]

Saved to demo_output_v1.txt


---
## V2 Model — 6 Topics · 5-Bucket VIX/EPU

Loads from `booth_results_v2/models/trump_gpt2_v2/`

In [4]:
MODEL_DIR_V2 = 'booth_results_v2/models/trump_gpt2_v2'
CUTOFFS_V2  = 'booth_results_v2/results/bucket_cutoffs_v2.json'

tok2, mdl2, dev2 = load_model(MODEL_DIR_V2)

with open(CUTOFFS_V2) as f:
    cuts2 = json.load(f)

VIX_LABELS_V2 = ['VIX_VERY_LOW', 'VIX_LOW', 'VIX_MED', 'VIX_HIGH', 'VIX_VERY_HIGH']
EPU_LABELS_V2 = ['EPU_VERY_LOW', 'EPU_LOW', 'EPU_MED', 'EPU_HIGH', 'EPU_VERY_HIGH']
TOPICS_V2     = [0, 1, 2, 3, 4, 5]

TOPIC_NAMES_V2 = {
    0: 'Fake News & Witch Hunt',
    1: 'Crooked Democrats',
    2: 'America First',
    3: 'MAGA Endorsements',
    4: 'Make America Great Again',
    5: 'Rallies & Events',
}

print(f'VIX cutoffs: {[round(x,2) for x in cuts2["vix"]]}')
print(f'EPU cutoffs: {[round(x,2) for x in cuts2["epu"]]}')

Loaded booth_results_v2/models/trump_gpt2_v2 on mps
VIX cutoffs: [12.63, 14.91, 17.54, 22.66]
EPU cutoffs: [108.21, 124.94, 149.97, 218.06]


In [5]:
out_path = 'demo_output_v2.txt'
lines = []
lines.append('=' * 70)
lines.append('V2 MODEL — ALL VIX x EPU COMBINATIONS')
lines.append('=' * 70)

for vl in VIX_LABELS_V2:
    for el in EPU_LABELS_V2:
        lines.append(f'\n{"─"*60}')
        lines.append(f'  [{vl}] [{el}]')
        lines.append(f'{"─"*60}')
        for t in TOPICS_V2:
            text = generate(tok2, mdl2, dev2, vl, el, t)
            lines.append(f'  [{TOPIC_NAMES_V2[t]}]  {text}')
        print(f'  Done: [{vl}] [{el}]')

with open(out_path, 'w') as f:
    f.write('\n'.join(lines))
print(f'\nSaved to {out_path}')

  Done: [VIX_VERY_LOW] [EPU_VERY_LOW]
  Done: [VIX_VERY_LOW] [EPU_LOW]
  Done: [VIX_VERY_LOW] [EPU_MED]
  Done: [VIX_VERY_LOW] [EPU_HIGH]
  Done: [VIX_VERY_LOW] [EPU_VERY_HIGH]
  Done: [VIX_LOW] [EPU_VERY_LOW]
  Done: [VIX_LOW] [EPU_LOW]
  Done: [VIX_LOW] [EPU_MED]
  Done: [VIX_LOW] [EPU_HIGH]
  Done: [VIX_LOW] [EPU_VERY_HIGH]
  Done: [VIX_MED] [EPU_VERY_LOW]
  Done: [VIX_MED] [EPU_LOW]
  Done: [VIX_MED] [EPU_MED]
  Done: [VIX_MED] [EPU_HIGH]
  Done: [VIX_MED] [EPU_VERY_HIGH]
  Done: [VIX_HIGH] [EPU_VERY_LOW]
  Done: [VIX_HIGH] [EPU_LOW]
  Done: [VIX_HIGH] [EPU_MED]
  Done: [VIX_HIGH] [EPU_HIGH]
  Done: [VIX_HIGH] [EPU_VERY_HIGH]
  Done: [VIX_VERY_HIGH] [EPU_VERY_LOW]
  Done: [VIX_VERY_HIGH] [EPU_LOW]
  Done: [VIX_VERY_HIGH] [EPU_MED]
  Done: [VIX_VERY_HIGH] [EPU_HIGH]
  Done: [VIX_VERY_HIGH] [EPU_VERY_HIGH]

Saved to demo_output_v2.txt


---
## Side-by-Side Comparison

Enter a specific VIX/EPU value and compare v1 vs v2 output.

In [6]:
# ── Set your values here ──────────────────────────
VIX_VALUE = 25.0
EPU_VALUE = 200.0
# ───────────────────────────────────────────────────

def bucket_v1(vix, epu, cuts):
    vl = 'VIX_LOW' if vix <= cuts['vix'][0] else ('VIX_MED' if vix <= cuts['vix'][1] else 'VIX_HIGH')
    el = 'EPU_LOW' if epu <= cuts['epu'][0] else ('EPU_MED' if epu <= cuts['epu'][1] else 'EPU_HIGH')
    return vl, el

def bucket_v2(vix, epu, cuts):
    c = cuts['vix']
    vl = ('VIX_VERY_LOW' if vix <= c[0] else 'VIX_LOW' if vix <= c[1] else
          'VIX_MED' if vix <= c[2] else 'VIX_HIGH' if vix <= c[3] else 'VIX_VERY_HIGH')
    c = cuts['epu']
    el = ('EPU_VERY_LOW' if epu <= c[0] else 'EPU_LOW' if epu <= c[1] else
          'EPU_MED' if epu <= c[2] else 'EPU_HIGH' if epu <= c[3] else 'EPU_VERY_HIGH')
    return vl, el

vl1, el1 = bucket_v1(VIX_VALUE, EPU_VALUE, cuts1)
vl2, el2 = bucket_v2(VIX_VALUE, EPU_VALUE, cuts2)

lines = []
lines.append(f'Input: VIX={VIX_VALUE}  EPU={EPU_VALUE}')
lines.append(f'V1 bucket: [{vl1}] [{el1}]')
lines.append(f'V2 bucket: [{vl2}] [{el2}]')
lines.append(f'\n{"="*70}')
lines.append('V1 OUTPUT')
lines.append('='*70)
for t in TOPICS_V1:
    text = generate(tok1, mdl1, dev1, vl1, el1, t)
    lines.append(f'  [Topic {t}] {text}')

lines.append(f'\n{"="*70}')
lines.append('V2 OUTPUT')
lines.append('='*70)
for t in TOPICS_V2:
    text = generate(tok2, mdl2, dev2, vl2, el2, t)
    lines.append(f'  [{TOPIC_NAMES_V2[t]}]  {text}')

out_path = f'demo_comparison_VIX{VIX_VALUE}_EPU{EPU_VALUE}.txt'
with open(out_path, 'w') as f:
    f.write('\n'.join(lines))

print('\n'.join(lines))
print(f'\nSaved to {out_path}')

Input: VIX=25.0  EPU=200.0
V1 bucket: [VIX_HIGH] [EPU_HIGH]
V2 bucket: [VIX_VERY_HIGH] [EPU_HIGH]

V1 OUTPUT
  [Topic 0] https://www. washingtonexaminer.com/opinion /hunter-biden-trump-wins-primary-michigan-by-10-points/2024/12/26/hunter-bidens-latest-promises-were-bashing-and-unfair-to-trump/article_1d4c78f-5ae5-
  [Topic 1] This is what America is all about. https://t.co/HVlSzpT4XU https://twitter.com/realDonaldTrump/status/12186680989835981 … pic.twitter.twitter iaE2kS4Q4XSj.twitter… pic iaHk5dWQI7oG—@
  [Topic 2] https://www. DonaldJTrump.com/news/2024/ 09/cnn-poll-trump-has-the-gutsy-line-as-gop-primary-primary/ … pic.twitter.com /jZbXDvvLsTQr – via @BreitbartNews. Thank you! https://twitter.co/B
  [Topic 3] [QuickTime Video] https://www. youtube.com/live/aJUxDQg2W4I?s i=OiRQcFpR_I9g0Oo4y5Wl0w7NlqWQ_n7SjWYZV-Rv2IyXKF5Zy0

V2 OUTPUT
  [Fake News & Witch Hunt]  FoxNews has just apologized to me for the terrible reporting they have done of the recent Witch Hunt. That’s because they d